# 03 — Preference Dataset Construction

Build pairwise preference data from collected trajectories, simulating human preferences using a deterministic scoring function. Extract trajectory feature vectors for reward model training.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pickle
import random
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
TRAJ_PATH = DATA_DIR / 'trajectories.pkl'
PAIRS_PATH = DATA_DIR / 'preference_pairs.pkl'
N_PAIRS = 1000
SEED = 42

assert TRAJ_PATH.exists(), f'Run notebook 02 first! Missing {TRAJ_PATH}'
random.seed(SEED)
np.random.seed(SEED)
print('Setup complete.')

In [ ]:
# Load trajectories
with open(TRAJ_PATH, 'rb') as f:
    trajectories = pickle.load(f)
print(f'Loaded {len(trajectories)} trajectories.')

In [ ]:
def preference_score(traj: dict) -> float:
    """
    Simulated human preference score.
    Rewards: high return + landing bonus - fuel penalty (proxy for graceful landing).
    """
    total_return = traj['total_return']
    landing_bonus = 50.0 if traj['landed'] else 0.0
    # Fuel proxy: actions 2 (main engine) incur cost
    fuel_use = np.sum(traj['actions'] == 2)  # main engine = action 2
    fuel_penalty = 0.2 * fuel_use
    return total_return + landing_bonus - fuel_penalty


def extract_features(traj: dict) -> np.ndarray:
    """
    Extract a fixed-size feature vector from a trajectory.
    Features (14-dim):
      - mean state (8 dims)
      - std state (8 dims) -- captures volatility
      - total return (1)
      - episode length (1)
      - fraction of main engine use (1)
      - landed flag (1)
    Total: 20 dims
    """
    states = traj['states']          # (T, 8)
    actions = traj['actions']        # (T,)
    mean_state = states.mean(axis=0) # (8,)
    std_state = states.std(axis=0)   # (8,)
    total_return = np.array([traj['total_return']])
    ep_len = np.array([len(actions)], dtype=np.float32)
    main_engine_frac = np.array([np.sum(actions == 2) / max(len(actions), 1)])
    landed = np.array([float(traj['landed'])])
    return np.concatenate([mean_state, std_state, total_return, ep_len, main_engine_frac, landed]).astype(np.float32)

# Verify feature dim
sample_feat = extract_features(trajectories[0])
FEAT_DIM = len(sample_feat)
print(f'Feature dimension: {FEAT_DIM}')
print(f'Sample features: {sample_feat}')

In [ ]:
# Pre-compute scores and features
scores = [preference_score(t) for t in trajectories]
features = [extract_features(t) for t in trajectories]

print(f'Score stats: mean={np.mean(scores):.1f}, std={np.std(scores):.1f}, '
      f'min={np.min(scores):.1f}, max={np.max(scores):.1f}')

In [ ]:
# Build preference pairs
indices = list(range(len(trajectories)))
pairs = []

for _ in range(N_PAIRS):
    i, j = random.sample(indices, 2)
    label = 1 if scores[i] > scores[j] else 0  # 1 if A preferred
    pairs.append({
        'feat_A': features[i],
        'feat_B': features[j],
        'label': label,
        'score_A': scores[i],
        'score_B': scores[j],
    })

label_dist = np.mean([p['label'] for p in pairs])
print(f'Built {len(pairs)} preference pairs.')
print(f'Label distribution: A preferred {label_dist:.2%}, B preferred {1-label_dist:.2%}')

In [ ]:
# Save preference pairs
with open(PAIRS_PATH, 'wb') as f:
    pickle.dump(pairs, f)

# Also save feature dim for reward model
meta = {'feat_dim': FEAT_DIM, 'n_pairs': len(pairs)}
with open(DATA_DIR / 'dataset_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print(f'Saved {len(pairs)} pairs to {PAIRS_PATH}')
print(f'Feature dim={FEAT_DIM} saved to dataset_meta.pkl')

In [ ]:
# Visualise preference score distribution
import matplotlib.pyplot as plt

score_gaps = [abs(p['score_A'] - p['score_B']) for p in pairs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scores, bins=40, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Preference Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Trajectory Preference Score Distribution')

axes[1].hist(score_gaps, bins=30, color='salmon', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('|Score_A - Score_B|')
axes[1].set_ylabel('Count')
axes[1].set_title('Score Gap per Preference Pair')

plt.tight_layout()
plt.savefig('../checkpoints/preference_data_stats.png', dpi=100)
plt.show()